In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import TransformedTargetRegressor

df = pd.read_csv("../data/processed/car_price_cleaned.csv")

features = [
    "name", "year", "km_driven", "fuel",
    "seller_type", "transmission", "owner"
]

X = df[features]
y = df["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = ["year", "km_driven"]
categorical_features = [
    "name", "fuel", "seller_type", "transmission", "owner"
]

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 2861
Testing records: 716


In [2]:
models = {
    "Random Forest Log Target": RandomForestRegressor(
        n_estimators=250,
        max_depth=25,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees Log Target": ExtraTreesRegressor(
        n_estimators=250,
        max_depth=25,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = []
tuned_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    regressor = TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    regressor.fit(X_train, y_train)
    predictions = regressor.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

    tuned_models[name] = regressor

display(pd.DataFrame(results).sort_values("MAE").round(3))

,Model,MAE,RMSE,R2
1,Extra Trees Log Target,165636.206,421322.395,0.449
0,Random Forest Log Target,167412.215,415657.085,0.464


In [3]:
comparison = pd.DataFrame(results)

print("Original Random Forest")
print("MAE: 174156.83")
print("RMSE: 404521.21")
print("R2: 0.4920")

print("\nLog-target models:")
display(comparison.round(3))

Original Random Forest
MAE: 174156.83
RMSE: 404521.21
R2: 0.4920

Log-target models:


,Model,MAE,RMSE,R2
0,Random Forest Log Target,167412.215,415657.085,0.464
1,Extra Trees Log Target,165636.206,421322.395,0.449
